# Build a Knowledge Graph for Your BGP Network — the guided version

*A warm, step-by-step lab for network engineers. You bring the BGP. We bring the graph.*

You already operate the largest graph ever built. It is called **the internet**, and the
protocol that stitches it together is **BGP**. Every autonomous system is a **node**. Every
session between two of them is an **edge**. And a route? A route is a **path across that
graph** — an `AS_PATH` is literally a list of the nodes a packet's reachability travelled
an `AS_PATH` is literally a list of the ASes a route was advertised through. You have been
reasoning about a graph your entire career; nobody just called it that.

A **knowledge graph** is that *same machine* — except **you** choose the nodes. And that one
freedom changes everything, because it lets you add the single node BGP was never designed to
hold: the **business service** that loses reachability when a session drops.

By the end of this notebook you will have built, from an empty page, a small graph that
answers the question no `show ip bgp summary` can:

> *"The eBGP session to our upstream just went down. Which business service is now at risk?"*

and watched it print **`Customer Portal`** — a fact that lives in nobody's routing table.

### First, calm the nerves: this is **not** machine learning
No training. No model. No embeddings. No AI guessing in the dark. A knowledge graph is just
**structured facts** (things, and the named links between them) plus **queries** that walk
those links. Everything is **deterministic and inspectable** — run it twice, get the same
answer, and you can point at every node that produced it. If you can read a BGP table, you can
read every line in this lab.

### What you need
Nothing. This runs in **Google Colab** with zero setup, using plain **Python + networkx +
matplotlib** (both already installed in Colab). No database, no Docker, no credentials. Press
*Runtime → Run all*, or run each cell in order with **Shift+Enter**.


## The whole vocabulary — a handful of words, each one a BGP thing you already know

Before any code, here is the *entire* mental model. Everything after this is these ideas,
repeated. And notice: you don't have to learn what any of them *mean* — you only have to learn
which BGP thing each one maps onto.

| Graph word | Plain meaning | The BGP thing it already is |
|---|---|---|
| **node** | a thing | a router (which belongs to an AS), a prefix, a service |
| **edge** | a named, directed link between two nodes | a session (a peering), "this router peers with that one" |
| **label** | a node's *type* | `Router`, `Prefix`, `Service` — like "is this an edge router vs a core router" |
| **property** | a fact stored *on* a node or edge | `state='down'`, `type='eBGP'`, `asn=65001` |
| **traversal** | walking edges to answer a question | following a route across sessions — you'll do it by hand, on purpose |

Two BGP-specific words ride along the whole way: an **AS / ASN** is one administrative routing
domain identified by a number; **eBGP** is a session that crosses an AS boundary, while **iBGP**
stays inside one AS. Hold those, and the rest is just your day job wearing a new hat.

One idea deserves a spotlight, because it is the whole trick: **a fact can live on an edge, not
just a node.** "The session is down" is not really a property of either router — it is a
property of the *session between them*. BGP knows this in its bones (a neighbor's state is
`Established`, or it isn't). In a graph you write it down literally: the failure lives **on the
edge**. Keep an eye out for that moment — it is where a topology diagram turns into something
you can *query*.


## Setup — one import, one empty graph

**Theory (10 seconds).** `networkx` is a tiny Python library for building graphs in memory. We
will use a **`DiGraph`** — a *directed* graph, where every edge has a direction, an arrow. That
matters to us: `edge-rtr-01 → upstream-isp-01` ("we peer *toward* the upstream") is a different
statement from the reverse. BGP is full of directional truth — who *advertises* to whom, which
way a prefix flows — so directed edges are the honest choice.

Run the next cell. It imports the tools and hands you an empty graph to fill.


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Both libraries ship pre-installed in Colab — nothing to pip install.
# A DiGraph is a *directed* graph: every edge is an arrow, from one node to another.
G = nx.DiGraph()

print("Empty graph ready:", G)
print("Nodes:", G.number_of_nodes(), " Edges:", G.number_of_edges())

**Checkpoint.** You should see `Empty graph ready` and `Nodes: 0  Edges: 0`. That empty `G` is
your blank whiteboard. Every step from here just adds nodes and edges to it.


## Step 1 — Routers and their roles

**Theory.** A router's *role* is not the same as the router. The same box can be an **edge**
router (it holds an eBGP session out to another AS) or a **core** router (it only speaks iBGP
inside your AS). You already think this way — "is this the box that peers with the transit
provider?" is a question about a *role*, not a serial number. And each router lives in an AS,
identified by its **ASN**.

So in the graph, `role` and `asn` are **properties** on the router node. We add three routers:

- **edge-rtr-01** — our edge router, in our AS `65001`, the box that peers out to the world.
- **upstream-isp-01** — our transit provider, a *different* AS (`701`). Reaching the rest of the
  internet runs through it.
- **core-rtr-01** — an internal core router, also in AS `65001`, speaking only iBGP.

Here is the design tension we're deliberately baking in: our reachability to the outside world
**hangs off a single eBGP session.** In a real network you'd want two upstreams, precisely so
one session loss can't cut you off. We model one on purpose — so the failure, and its blast
radius, shows up cleanly in a single edge you can point at.

Notice we are *not* dumping the BGP table in here. A node per router — not a node per route.
We store the *shape* of the design, and we'll come back to that principle by name later.


In [ ]:
# add_node(name, **properties). The name is a unique handle; label/asn/role are facts on it.
G.add_node("edge-rtr-01",     label="Router", asn=65001, role="edge")
G.add_node("upstream-isp-01", label="Router", asn=701,   role="transit")
G.add_node("core-rtr-01",     label="Router", asn=65001, role="core")

for n, d in G.nodes(data=True):
    if d.get("label") == "Router":
        print(f'{n}: AS{d["asn"]}, role={d["role"]}')

**Checkpoint.** Three routers print with their AS and role: `edge-rtr-01` (AS65001, edge),
`upstream-isp-01` (AS701, transit), `core-rtr-01` (AS65001, core). They're floating unconnected
for now — the next step wires them together, and that wire is where the story turns.


## Step 2 — Sessions: the edge carries the state (this is the pivotal step)

**Theory.** Here is the idea the whole lab pivots on. When a BGP session drops, *where* does
that fact belong? Not on our router (it's fine). Not on the upstream (it's fine too). It belongs
on the **session between them**. BGP already agrees with you: a neighbor is either `Established`
or it is not — that state describes the *adjacency*, not either endpoint.

So a session doesn't need its own node — it *is* the **edge** between two routers. We add edges
with a `rel` of `PEERS_WITH`, and hang two properties on each: `type` (`eBGP` if it crosses an
AS boundary, `iBGP` if it stays inside ours) and `state` (`up`/`down`). Read
`edge-rtr-01 --PEERS_WITH(eBGP, down)--> upstream-isp-01` literally: *"we peer, externally, with
the upstream — and that session is down."*

The wiring:

- `edge-rtr-01 → upstream-isp-01` is an **eBGP** session, and it is **down** — our path to the
  world just broke. This one edge is the incident.
- `edge-rtr-01 → core-rtr-01` is an **iBGP** session, and it is **up** — the inside of our AS is
  perfectly healthy. Hold onto that: an unrelated, *up* session is sitting right next to the
  broken one, and a good blast-radius query must **not** blame it. We'll prove that it doesn't.


In [ ]:
# add_edge(source, target, **properties). The session IS the edge; state is a property ON it.
G.add_edge("edge-rtr-01", "upstream-isp-01", rel="PEERS_WITH", type="eBGP", state="down")  # <-- the incident
G.add_edge("edge-rtr-01", "core-rtr-01",     rel="PEERS_WITH", type="iBGP", state="up")

for u, v, d in G.edges(data=True):
    print(f'{u} --{d["rel"]}({d["type"]}, {d["state"]})--> {v}')

**Checkpoint.** Two `PEERS_WITH` edges print, and exactly one reads `down`: the
`edge-rtr-01 --PEERS_WITH(eBGP, down)--> upstream-isp-01` line. That single edge, with a property
on it, is the 3 AM event — sitting in your graph as a fact you can now query against.


## See it — draw the graph

**Theory.** Text is great during an incident, but to *understand* a design you want the
picture. We'll colour nodes by their **label** (routers one colour, prefixes and services
others) and draw the one **down** session as a dashed red arrow so the failure jumps out. This
is the same instinct as a topology diagram — except this diagram is generated *from the data*,
so it can never drift out of sync with the truth.


In [ ]:
def draw(G, title="BGP knowledge graph"):
    colors = {"Router": "#3aa0ff", "Prefix": "#0f7f74", "Service": "#c8702a"}
    node_colors = [colors.get(G.nodes[n].get("label"), "#cccccc") for n in G]
    pos = nx.spring_layout(G, seed=7)   # fixed seed => stable, repeatable layout

    plt.figure(figsize=(10, 7))
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1900, alpha=0.92)
    nx.draw_networkx_labels(G, pos, font_size=9)

    down  = [(u, v) for u, v, d in G.edges(data=True) if d.get("state") == "down"]
    other = [(u, v) for u, v, d in G.edges(data=True) if (u, v) not in down]
    nx.draw_networkx_edges(G, pos, edgelist=other, arrows=True, arrowsize=16, edge_color="#8894a0")
    nx.draw_networkx_edges(G, pos, edgelist=down,  arrows=True, arrowsize=16,
                           edge_color="#d34b3f", width=2.0, style="dashed")
    nx.draw_networkx_edge_labels(G, pos, font_size=7,
        edge_labels={(u, v): d.get("rel", "") for u, v, d in G.edges(data=True)})

    plt.axis("off"); plt.title(title); plt.tight_layout(); plt.show()

draw(G)

**Reading the picture.** You should see the three routers (blue) with `PEERS_WITH` arrows
between them, and one **dashed red** arrow from `edge-rtr-01` to `upstream-isp-01` — the down
eBGP session. This is still just a topology. It becomes a *knowledge* graph in the next step,
when we add the things a topology diagram has never carried: the prefix, and the business
behind it.


## Step 3 — Prefix lineage: tie a route to the session it rides on

**Theory.** On a topology diagram a prefix is just a label floating near a router. In a
knowledge graph a prefix gets **provenance** — a traceable story of *how the outside world
reaches it*. And for BGP the crucial part of that story is: **which session carries it.**

We add one **Prefix** node, `203.0.113.0/24` (the public block our service lives behind), and
wire it with an `ADVERTISES` edge from `edge-rtr-01`. The trick is a property on that edge:
`via="upstream-isp-01"` — *"we advertise this prefix out, and it reaches the world **via** that
specific session."* That `via` tag is what lets the blast radius flag only the prefixes affected
by **this** down session, and leave alone anything reachable another way.

The **stop-and-think rule** you're using here, without me naming it yet: store *summarized
operational truth and dependency lineage* — a node for the prefix, and the one session it
depends on — and leave the 900,000-route global table where it lives, in the router. The graph
holds the *shape* of the dependency, not the telemetry firehose.


In [ ]:
G.add_node("203.0.113.0/24", label="Prefix", origin_as=65001)
G.add_edge("edge-rtr-01", "203.0.113.0/24", rel="ADVERTISES", via="upstream-isp-01")

print("Prefix facts:", G.nodes["203.0.113.0/24"])
print("How the prefix is advertised:")
for u, v, d in G.in_edges("203.0.113.0/24", data=True):
    print(f'  {u} --{d["rel"]}(via {d.get("via")})--> {v}')

**Checkpoint.** The prefix now has one incoming edge: `edge-rtr-01 --ADVERTISES(via
upstream-isp-01)--> 203.0.113.0/24`. The route has a *story* now, and — crucially — that story
names the exact session it depends on. But a route with a story still isn't a business fact —
that's the final, missing node, next.


## Step 4 — The business service: the node BGP never had

**Theory.** This is the node your routers were never designed to hold, and the reason the whole
lab exists. BGP knows sessions, prefixes, AS paths, communities, local-pref. It has never once
known that `203.0.113.0/24` is where the **Customer Portal** lives, and that when that prefix
loses its path to the world, **revenue stops**.

So we add a **Service** node, `Customer Portal`, with a `criticality` property, and wire it to
the prefix with a `SUPPORTS` edge: *"this prefix supports that service."* One edge — and now a
routing fact and a business fact are welded together in a place you can query. No `show` command
holds this link. It has never lived anywhere except tribal knowledge. You're about to make it a
first-class, walkable fact.


In [ ]:
G.add_node("Customer Portal", label="Service", criticality="critical")
G.add_edge("203.0.113.0/24", "Customer Portal", rel="SUPPORTS")

print("Graph now has", G.number_of_nodes(), "nodes and", G.number_of_edges(), "edges.")
print("The load-bearing link:", [f'{u} -SUPPORTS-> {v}'
      for u, v, d in G.edges(data=True) if d.get("rel") == "SUPPORTS"])
draw(G, title="BGP knowledge graph — now with the business service")

**Checkpoint.** The graph has grown to include an orange `Service` node, `Customer Portal`,
joined by a `SUPPORTS` edge to the prefix. In the redrawn picture you can *trace with your
finger*: down session → edge-rtr-01 → prefix → Customer Portal. In the next step we make the
computer trace it for you.


## Step 5 — The 3 AM question: blast radius as a traversal

**Theory.** Everything so far was setup for this. A **traversal** is just walking edges to
answer a question — the exact thing a router does when it resolves a path, except *you* decide
the walk. Our walk answers:

> *"For any **eBGP** session that is **down**, which business services are at risk?"*

The route the walk takes:

1. find a `PEERS_WITH` edge whose `type` is `eBGP` and whose `state` is `down` — note both the
   router and the neighbor it names;
2. from that router, hop to the **prefixes** it `ADVERTISES` **`via` that exact neighbor** (this
   is where the `via` tag earns its keep — a prefix reachable another way is skipped);
3. from each prefix, hop to the **services** it `SUPPORTS`.

Every hop is one edge. Two things make this a *model* and not a drawing. First, the query is
**conditioned on session state** — flip the eBGP edge to `up` and step 1 finds nothing, so the
whole walk returns nothing. Second, it is **scoped by `via`** — the healthy iBGP session to
`core-rtr-01` never enters the walk, because nothing is tagged as depending on it. Run it.


In [ ]:
def blast_radius(G):
    # Services put at risk by any eBGP session that is currently down.
    at_risk = []
    for rtr, nbr, d in G.edges(data=True):
        # 1) a PEERS_WITH edge that is eBGP and DOWN
        if d.get("rel") == "PEERS_WITH" and d.get("type") == "eBGP" and d.get("state") == "down":
            # 2) prefixes this router advertises specifically VIA that neighbor
            for _, pfx, d2 in G.out_edges(rtr, data=True):
                if d2.get("rel") != "ADVERTISES" or d2.get("via") != nbr:
                    continue
                # 3) services those prefixes support
                for _, svc, d3 in G.out_edges(pfx, data=True):
                    if d3.get("rel") == "SUPPORTS":
                        at_risk.append((rtr, nbr, pfx, svc))
    return at_risk

hits = blast_radius(G)
for rtr, nbr, pfx, svc in hits:
    print(f"{rtr} eBGP to {nbr} DOWN  ->  {pfx}  ->  AT RISK: {svc}")
if not hits:
    print("nothing at risk")

The router only ever told you a session dropped — one line in a log. Your graph just told you
the **Customer Portal is at risk**, and showed its work: the whole path from the down session to
the revenue. Notice what it did *not* say — the healthy iBGP session to `core-rtr-01` never came
up, because no prefix was tagged as riding on it. You got that precision because *you* added the
one node BGP never had, and tagged exactly which session each prefix depends on. That is the
entire pitch of a knowledge graph, and you just built it from an empty page.


## Step 6 — What-if: repair the session, then break it again

**Theory.** Because the failure lives on an edge as a plain property, you can *change* it and
re-ask — running "what if this fails?" (or "what if I fix it?") on a **model**, safely, with no
one paged and no maintenance window. Flip the eBGP session to `up`: the blast radius clears.
Flip it back to `down`: the Customer Portal returns. This is the humble beginning of
pre-incident planning — you can ask the graph what *would* break before it does.


In [ ]:
# You can read and write an edge's properties directly by [source][target].
G["edge-rtr-01"]["upstream-isp-01"]["state"] = "up"     # repair the session
print("After repair:   ", blast_radius(G) or "nothing at risk")

G["edge-rtr-01"]["upstream-isp-01"]["state"] = "down"   # break it again
print("After re-break: ", [svc for *_, svc in blast_radius(G)])

**Checkpoint.** After the repair you should see `nothing at risk`; after re-breaking it,
`['Customer Portal']` comes back. Same graph, same query — only the state on one edge changed.
The answer *responded*. That is what makes it a model rather than a drawing.


## Your turn #1 — a second service on the same prefix

Real prefixes rarely support just one thing. Suppose a **Payments Webhook** service also rides
on `203.0.113.0/24`. Add it, wire it with a `SUPPORTS` edge, and re-run `blast_radius`. You
should now see **two** services surface from the same down session — because one edge of extra
truth is all it takes.

Fill in the cell below (there's a `# TODO`), then run it. The solution follows if you want to
check.


In [ ]:
# TODO: add a "Payments Webhook" Service node (criticality="high"),
#       then a SUPPORTS edge from "203.0.113.0/24" to it.

# G.add_node(...)
# G.add_edge(...)

for rtr, nbr, pfx, svc in blast_radius(G):
    print(f"AT RISK: {svc}  (via {pfx}, {nbr} session down)")

<details><summary><b>Solution — click to reveal</b></summary>

```python
G.add_node("Payments Webhook", label="Service", criticality="high")
G.add_edge("203.0.113.0/24", "Payments Webhook", rel="SUPPORTS")

for rtr, nbr, pfx, svc in blast_radius(G):
    print(f"AT RISK: {svc}  (via {pfx}, {nbr} session down)")
```

Now both `Customer Portal` **and** `Payments Webhook` come back from the one down eBGP session.
The blast radius grew the instant you told the graph one more true thing — you didn't touch the
query at all.
</details>


In [ ]:
# (Solution, applied — so the rest of the notebook has both services in the graph.)
G.add_node("Payments Webhook", label="Service", criticality="high")
G.add_edge("203.0.113.0/24", "Payments Webhook", rel="SUPPORTS")
print("Services at risk now:", sorted({svc for *_, svc in blast_radius(G)}))

## Quiet reveal — you have been writing an *ontology*

Time to name the thing you've been doing. Every step, you made two very different kinds of
decision, and it's worth seeing the seam between them:

- *"A `Router` has an `asn`. A `Prefix` is `ADVERTISES`-ed by a `Router` `via` a session. A
  `Service` is `SUPPORTS`-ed by a `Prefix`."* — these are the **rules**: which node types exist,
  which edges are allowed, what shape a valid fact takes. That is the **schema**. Its fancy name
  is an **ontology** — and here's the friendly definition: *an ontology is the RFC of your graph,
  the agreed vocabulary written down as structure.* You already trust RFC 4271 to say what a
  valid BGP session is; an ontology does the same job for your graph.

- *"This particular router is `edge-rtr-01`, ASN `65001`, and its eBGP session is `down`."* —
  these are the **instances**: the actual data.

A rule of thumb that keeps the two straight forever: **if it has a hostname, an IP, or an ASN,
it is data (an instance), never schema.** `Router` is schema; `edge-rtr-01` is data.
`PEERS_WITH` is schema; "this session is down" is data. Keep the vocabulary small and stable;
let the instances be many. That single discipline is the difference between a graph you can grow
for years and a swamp you abandon in a month.


## A peek at the real thing (1/2) — the same routers in Neo4j + Cypher

We used networkx so you could run everything with zero setup. But the *shapes* you built are
exactly what you'd build in a real graph database like **Neo4j**, whose query language is
**Cypher**. Cypher is worth a glance because it reads almost like the arrows we've been drawing
— `(node)-[:EDGE]->(node)`.

Here is **Step 2 (the routers and the session)** as Cypher. `MERGE` means "find this node or
create it if missing" (so re-running is safe); `SET` assigns properties. This is *reference
only* — you don't run it here, it just shows you the same idea in the grown-up tool:

```cypher
MERGE (edge:Router {id: 'edge-rtr-01'})
SET   edge.asn = 65001, edge.role = 'edge';

MERGE (isp:Router {id: 'upstream-isp-01'})
SET   isp.asn = 701, isp.role = 'transit';

// the failure, as a property on the relationship — same as our edge
MERGE (edge)-[r:PEERS_WITH]->(isp)
SET   r.type = 'eBGP', r.state = 'down';
```

(Note the `MERGE ... SET` idiom on the relationship, mirroring the nodes: `MERGE` finds-or-
creates the *one* session edge, and `SET` assigns its state — so re-running with a new state
updates that edge instead of piling up duplicates.)

See it? `(edge)-[:PEERS_WITH {type:'eBGP', state:'down'}]->(isp)` is character-for-character the
same statement as our `G.add_edge("edge-rtr-01", "upstream-isp-01", rel="PEERS_WITH",
type="eBGP", state="down")`. Same nodes, same edge, same fact-on-the-edge — one just happens to
live in a database that scales to millions of nodes.


## A peek at the real thing (2/2) — the 3 AM question in Cypher

Our `blast_radius` walk is a 15-line Python function. In Cypher, that same traversal is four
lines, because in a graph database you **draw the shape you're looking for** and let the engine
find every match — no manual loops:

```cypher
MATCH (rtr:Router)-[s:PEERS_WITH {type:'eBGP', state:'down'}]->(nbr:Router)
MATCH (rtr)-[a:ADVERTISES]->(prefix:Prefix)  WHERE a.via = nbr.id
MATCH (prefix)-[:SUPPORTS]->(svc:Service)
RETURN rtr.id AS router, nbr.id AS neighbor,
       collect(DISTINCT svc.id) AS services_at_risk;
```

Read the first line as a sentence: *"match a router whose eBGP session is down, and the neighbor
on the far end."* The second line is our `via` scope — *"...that advertises a prefix via that
exact neighbor."* That's the same steps 1 and 2 you wrote in Python — the pattern you `MATCH`
**is** the traversal. Run it against a real dataset and `edge-rtr-01` comes back with
`Customer Portal` (and `Payments Webhook`) as the services at risk. Different engine; identical
thinking. If you can read the Python above, you can read the Cypher — you already speak this
language.


## Your turn #2 — make the query respond to a *different* failure

Right now a single upstream carries everything. Real edges have more than one. Let's add a
**second** upstream that is perfectly healthy, prove the query leaves its service alone, then
break it and watch the service surface — all without touching the query:

1. add a second transit router `isp-secondary` (AS `64500`), and an **eBGP** session from
   `edge-rtr-01` to it that is **up**;
2. add a prefix `198.51.100.0/24` advertised **`via` isp-secondary**, supporting a new service
   **Partner API**;
3. re-run `blast_radius` — Partner API should **not** appear (its session is up);
4. now set the `edge-rtr-01 → isp-secondary` session to `down` and re-run — Partner API
   **appears**.

This is the whole point of Step 6, in your own hands: the answer follows the state, and it stays
scoped to the session that actually failed. Fill in the `# TODO`s, then run.


In [ ]:
# TODO 1: add Router "isp-secondary" (asn=64500, role="transit"), then:
#   - an eBGP session that is UP:  edge-rtr-01 -> isp-secondary  (type="eBGP", state="up")
#   - Prefix "198.51.100.0/24" advertised VIA isp-secondary:  edge-rtr-01 -> prefix (via="isp-secondary")
#   - Service "Partner API" (criticality="medium") supported by that prefix

# G.add_node(...); G.add_edge(...); ...

print("With isp-secondary UP:  ", sorted({s for *_, s in blast_radius(G)}))

# TODO 2: break the second session (G["edge-rtr-01"]["isp-secondary"]["state"] = "down"),
#         then print blast_radius again to watch Partner API appear.
#         (The solution below demonstrates the DOWN case in full.)

<details><summary><b>Solution — click to reveal</b></summary>

```python
G.add_node("isp-secondary", label="Router", asn=64500, role="transit")
G.add_edge("edge-rtr-01", "isp-secondary", rel="PEERS_WITH", type="eBGP", state="up")
G.add_node("198.51.100.0/24", label="Prefix", origin_as=65001)
G.add_edge("edge-rtr-01", "198.51.100.0/24", rel="ADVERTISES", via="isp-secondary")
G.add_node("Partner API", label="Service", criticality="medium")
G.add_edge("198.51.100.0/24", "Partner API", rel="SUPPORTS")

print("With isp-secondary UP:  ", sorted({s for *_, s in blast_radius(G)}))
# -> Customer Portal, Payments Webhook  (Partner API is safe: its session is up)

G["edge-rtr-01"]["isp-secondary"]["state"] = "down"
print("With isp-secondary DOWN:", sorted({s for *_, s in blast_radius(G)}))
# -> now Partner API joins the list
```

Before you broke the session, Partner API was *in the graph* but *not at risk* — the query
correctly ignored a healthy path. The moment its session went down, the exact same query
surfaced it, and *only* it — Customer Portal's separate session was never touched. You didn't
teach the query about a second upstream; you just told the graph the truth, and the traversal
did the rest.
</details>


In [ ]:
# (Solution, applied — surfaces Partner API on failure, then restores health for later cells.)
G.add_node("isp-secondary", label="Router", asn=64500, role="transit")
G.add_edge("edge-rtr-01", "isp-secondary", rel="PEERS_WITH", type="eBGP", state="up")
G.add_node("198.51.100.0/24", label="Prefix", origin_as=65001)
G.add_edge("edge-rtr-01", "198.51.100.0/24", rel="ADVERTISES", via="isp-secondary")
G.add_node("Partner API", label="Service", criticality="medium")
G.add_edge("198.51.100.0/24", "Partner API", rel="SUPPORTS")

print("With isp-secondary UP:  ", sorted({s for *_, s in blast_radius(G)}))
G["edge-rtr-01"]["isp-secondary"]["state"] = "down"
print("With isp-secondary DOWN:", sorted({s for *_, s in blast_radius(G)}))
G["edge-rtr-01"]["isp-secondary"]["state"] = "up"   # restore health
print("Restored, at risk again:", sorted({s for *_, s in blast_radius(G)}))

## Make it yours — swap in *your* network

Now the moment it becomes your tool, not mine. Change the five values below to **one** eBGP
session, **one** upstream, **one** prefix, and **one** service *you* actually run. We add the
down session for you, so your service shows up. Run it, and watch **your own service name** come
back from `blast_radius` — proof that the machine you built understands your network, not just
the demo's.

Keep it to the smallest slice that answers one real question. You can always add one more node
tomorrow — that's the whole promise of a graph you can grow.


In [ ]:
# --- change these five values to your network ---
MY_ROUTER   = "edge-rtr-99"
MY_UPSTREAM = "my-isp"
MY_ASN      = 65099
MY_PREFIX   = "192.0.2.0/24"
MY_SERVICE  = "Booking API"
# -------------------------------------------------

G.add_node(MY_ROUTER,   label="Router", asn=MY_ASN, role="edge")
G.add_node(MY_UPSTREAM, label="Router", asn=64999,  role="transit")
G.add_node(MY_PREFIX,   label="Prefix", origin_as=MY_ASN)
G.add_node(MY_SERVICE,  label="Service", criticality="critical")

G.add_edge(MY_ROUTER, MY_UPSTREAM, rel="PEERS_WITH", type="eBGP", state="down")  # your modelled failure
G.add_edge(MY_ROUTER, MY_PREFIX,   rel="ADVERTISES", via=MY_UPSTREAM)
G.add_edge(MY_PREFIX, MY_SERVICE,  rel="SUPPORTS")

print(f"If {MY_ROUTER}'s eBGP session to {MY_UPSTREAM} fails, these services are at risk:")
for rtr, nbr, pfx, svc in blast_radius(G):
    if rtr == MY_ROUTER:
        print(f"  AT RISK: {svc}   (via {pfx})")

**Checkpoint.** Your own service name — `Booking API`, or whatever you typed — prints as *at
risk*. That is the moment the tool stopped being a tutorial and became yours. Every other
service you run is just four more lines away.


## Go deeper (optional) — the next-hop, one layer down

We kept a real BGP subtlety out of the core on purpose, but it's worth a callout because it's
the same shape one level down. A BGP route doesn't just need its *session* up — it needs its
**next-hop** to be *reachable*, and that reachability is resolved recursively by your IGP (OSPF
or IS-IS). A route can sit in the table looking perfectly valid and still be **unusable**
because the IGP path to its next-hop is gone.

In graph terms that's just one more node and one more hop: a `NextHop` node the prefix
`HAS_NEXT_HOP`, `DEPENDS_ON` an `IGPReachability` node with its own `state`. Add those, and your
blast radius can catch a second, sneakier failure mode — the route that's present but dead. That
is exactly where a **BGP graph meets an OSPF graph**: the BGP next-hop resolves through the OSPF
topology you modelled in the companion lab. One graph, two protocols. We leave it as the natural
next thing to build — the five words carry it without change.


## The one rule that keeps this from becoming a swamp

You'll be tempted to model everything. Don't. Here is the discipline, in one line:

> **Model the nouns of your design review, not the numbers of your monitoring dashboard.**

Routers, sessions, ASes, prefixes, communities, policies, services — the **nouns** you'd draw on
a whiteboard while arguing about a design — those belong in the graph. Per-prefix update
counters, session flap timers, CPU, the 900,000-route global table — those are the **numbers**;
leave them in the systems that already store them well, and let the graph *reference* them when
it needs to. The graph holds the *shape of the dependency*; your NMS holds the firehose. Keep
that line sharp and your graph stays queryable for years.

### Where to go next
- **The companion Neo4j lab** does this exact thing in a real graph database — same nodes, same
  edges, same 3 AM question — so the two Cypher peeks above become things you run.
- **Span two protocols.** Wire a BGP `NextHop` to an OSPF/IS-IS reachability node (the callout
  above) and "is this route actually usable?" becomes one more traversal.
- **Add the change layer.** Model a `ChangeEvent` node linked to what it touched, and "what
  changed right before this session flapped?" becomes one more traversal.


## You built a BGP knowledge graph

Look back at the distance. You started with an empty page and a handful of plain words. You
added routers and a down session — a topology. Then you tied a prefix to the exact session it
rides on, and added the one node BGP never had, the **business service** — and the topology
became *knowledge*. Finally you asked it the question no router can answer, and it printed
**Customer Portal**, then responded correctly every time you changed the world underneath it —
surfacing the right service, from the right failure, and leaving the healthy paths alone.

The important idea was never BGP, and never networkx. It's this: **operational truth has a
shape** — a service depends on a prefix, a prefix is advertised via a specific session, a router
peers with another router — and once that shape is a graph, impact analysis stops being tribal
knowledge and becomes a traversal. A runbook is a hammer. A graph you can query is the whole
toolbox.

You already operate the biggest graph on Earth. Now you know how to build the one that holds the
part BGP was never asked to remember. Go model one real service on your network, and ask it what
breaks.
